# Training template — gradient tracker

Copy this notebook into your own project. Fill in the three TODOs (your model,
your data, your loss). The tracker calls are already wired into a generic loop —
you don't change them. The two tools impose nothing else on how you code.

In [ ]:
import sys
sys.path.insert(0, "..")            # adjust to point at the repo's src/
import torch
from src.gradient_tracker import GradientTracker
from src.architecture import to_mermaid

## 1. TODO: your model — any `nn.Module`

In [ ]:
# model = YourModel(...)
# print(to_mermaid(model, input_shape=(1, <your_input_dim>)))

## 2. TODO: your data — any iterable of batches

In [ ]:
# train_loader = ...
# val_loader = ...   # optional

## 3. TODO: your loss — any callable returning a scalar to .backward()

In [ ]:
# def compute_loss(model, batch):
#     ...
#     return loss

## 4. Train — generic loop, tracker calls already in place

In [ ]:
tracker = GradientTracker(model, metric="l2_norm")
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

EPOCHS = 100
for epoch in range(EPOCHS):
    model.train()
    running = 0.0; nb = 0
    for batch in train_loader:
        optimizer.zero_grad()
        loss = compute_loss(model, batch)
        loss.backward()
        tracker.accumulate()            # after backward, before next zero_grad
        optimizer.step()
        running += float(loss); nb += 1
    train_loss = running / max(nb, 1)

    # optional validation
    val_loss = None
    # model.eval(); ... compute val_loss ...

    tracker.log_epoch(epoch, losses={"train": train_loss,
                                     **({"val": val_loss} if val_loss is not None else {})})
print("done")

## 5. Inspect

In [ ]:
tracker.plot_heatmap("gradient_heatmap.png")
tracker.plot_curves("gradient_curves.png")
tracker.plot_contributions("layer_contributions.png")